# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: EDA with `mlcroissant`

This notebook demonstrates how to load, inspect, and explore the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All dataset entities—record sets, fields, columns—are referenced by their `@id` as per best practices for Croissant packages.

### Dataset Source

Croissant Schema URL:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and prepare the `mlcroissant` Dataset for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load Dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset name:", metadata.name)
print("Description:", metadata.description)

# Optionally show publication or other metadata
print(f"Publication date: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Number of record sets: {len(metadata.record_set) if hasattr(metadata, 'record_set') else 0}")

## 2. Data Overview

Let's explore the available record sets, their `@id` values, and fields, following the Croissant package schema.

*For this dataset, we will print summary information for each RecordSet and its Fields using their `@id` identifiers.*

In [ ]:
# Extract record set information via @id
if not hasattr(metadata, 'record_set') or not metadata.record_set:
    # If the metadata.record_set is missing or empty, discover record sets instead
    print("No record sets found in metadata.record_set. Attempting to infer available record sets from the dataset.")
    discovered = list(dataset.record_sets())
    print("Discovered RecordSet @id values:")
    for rs in discovered:
        print(f"- {rs}")
    record_set_ids = discovered
else:
    record_set_list = metadata.record_set if isinstance(metadata.record_set, list) else [metadata.record_set]
    print("RecordSets declared in metadata.record_set:")
    record_set_ids = [r['@id'] if isinstance(r, dict) and '@id' in r else str(r) for r in record_set_list]
    for rsid in record_set_ids:
        print(f"- {rsid}")

first_set = record_set_ids[0] if record_set_ids else None

# Show Field @id-s for the first RecordSet
if first_set:
    record_set_obj = dataset.record_set(first_set)
    print(f"\nFields in RecordSet {first_set}:")
    for field in record_set_obj.fields:
        print(f"- Field @id: {field['@id']}  (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")


## 3. Data Extraction

We now load the record sets as Pandas DataFrames using the `@id` identifiers.

Below, data for each record set is loaded into a DataFrame (`dataframes[@id]`).  The first few columns and rows are shown for the main record set.


In [ ]:
# Prepare to extract data for all discovered record sets
dataframes = {}
for record_set in record_set_ids:
    print(f"\nLoading records from RecordSet @id: {record_set}")
    records = list(dataset.records(record_set=record_set))
    if not records:
        print("No records found for this record set.")
        continue
    dataframes[record_set] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set])} records. Columns:")
    print(dataframes[record_set].columns.tolist())
    display(dataframes[record_set].head(3))

# Pick first non-empty record set (for EDA below)
main_record_set_id = next((k for k, v in dataframes.items() if not v.empty), None)
if main_record_set_id:
    print(f"\nPrimary data will be explored from RecordSet: {main_record_set_id}")
else:
    print("No populated DataFrames available for EDA.")

## 4. Exploratory Data Analysis (EDA)

Now, let's perform some basic exploratory steps on the main record set. We'll:
- Select a numeric field by its `@id`
- Filter for values above a threshold
- Normalize the numeric field
- Group by a categorical field (if possible)

*All processing below uses the Croissant `@id` as the column identifier.*

In [ ]:
# EDA on the main record set
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Columns in primary DataFrame ({main_record_set_id}):\n{list(df.columns)}")

    # Try to pick a numeric field (@id) for demonstration: e.g., '@id': 'age' or similar, fallback to any numeric column
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric-type columns found: {numeric_candidates}")

    # Select first numeric field for examples
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]

        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"\nFiltered records with '{numeric_field_id}' > {threshold:.2f} (using @id): {len(filtered_df)} records")
        display(filtered_df.head(3))

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
        print(f"\nShowing normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

        # Attempt grouping by a field
        candidate_categorical = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])]
        group_field_id = candidate_categorical[0] if candidate_categorical else None
        if group_field_id:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}' (@id):")
            display(grouped.head())
        else:
            print("No suitable categorical field found to group by.")
    else:
        print("No numeric fields available for EDA in the main record set.")
else:
    print("No populated DataFrame to analyze.")

## 5. Visualization

We can visualize the distribution of a numeric field and relationships with a categorical field. Below, we create a histogram and a box plot using Matplotlib and Seaborn.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set_id is not None and numeric_candidates:
    numeric_field = numeric_candidates[0]

    # Histogram
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.xlabel(numeric_field + " (@id)")
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # Boxplot by first categorical (if available)
    if candidate_categorical:
        cat_field = candidate_categorical[0]
        plt.figure(figsize=(8,4))
        sns.boxplot(x=cat_field, y=numeric_field, data=df)
        plt.xlabel(cat_field + " (@id)")
        plt.ylabel(numeric_field + " (@id)")
        plt.title(f"{numeric_field} by {cat_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("No numeric data to plot.")

## 6. Conclusion

In this notebook, we've demonstrated step-by-step how to load and explore a FAIR-compliant dataset using [`mlcroissant`](https://pypi.org/project/mlcroissant/), referencing all data entities by their `@id`. We've shown how to:
 - Enumerate available record sets and fields,
 - Load data into pandas DataFrames using each record set's `@id`,
 - Perform statistical filtering and normalization by `@id`-labeled fields,
 - Visualize numerical and categorical distributions.

This approach establishes a reproducible, interoperable data science workflow based strictly on FAIR metadata and `mlcroissant`'s robust schema-handling.